In [ ]:
import os
import pandas as pd

# Base directory to start the recursive search
base_dir = '../data/'

# Target filenames
MMWAVE_FILENAME = 'mmwave_data.csv'
DISTANCE_FILENAME = 'distance_sensors.csv'
MERGED_FILENAME = 'merged_df.csv'

# Columns used for loop detection logic
box1_col = 'box1_active'
box5_col = 'box5_active'
# Single velocity column name
vel_col = 'velocity'

# Recursively walk through the base directory
for root, dirs, files in os.walk(base_dir):
    # Check if both required CSV files are present in the current directory's files
    if MMWAVE_FILENAME in files and DISTANCE_FILENAME in files:
        # If both files exist, this 'root' is the folder to process
        # 'root' here is equivalent to 'exp_path' in your old code
        current_processing_folder = root
        print(f"Processing folder: {current_processing_folder}")

        mmwave_path = os.path.join(current_processing_folder, MMWAVE_FILENAME)
        distance_path = os.path.join(current_processing_folder, DISTANCE_FILENAME)
        merged_path = os.path.join(current_processing_folder, MERGED_FILENAME)

        try:
            # Load both mmWave and distance sensor data
            mmwave_df = pd.read_csv(mmwave_path)
            distance_df = pd.read_csv(distance_path)

            # Convert timestamps to datetime format
            mmwave_df['timestamp'] = pd.to_datetime(mmwave_df['timestamp'])
            distance_df['timestamp'] = pd.to_datetime(distance_df['timestamp'])

            # Filter out points with zero velocity
            try:
                mmwave_df = mmwave_df[mmwave_df[vel_col] != 0]
            except KeyError as e:
                print(f"Velocity column '{vel_col}' not found in {mmwave_path}: {e}. Skipping velocity filtering.")

            # Find common frame_ids
            common_frames = set(mmwave_df['frame_id']).intersection(set(distance_df['frame_id']))
            if not common_frames:
                print(f"[!] No common frame_ids found between {mmwave_path} and {distance_path}. Skipping merge for this folder.")
                continue # Skip to the next folder

            mmwave_filtered = mmwave_df[mmwave_df['frame_id'].isin(common_frames)]
            distance_filtered = distance_df[distance_df['frame_id'].isin(common_frames)]
            distance_filtered = distance_filtered.drop_duplicates(subset='frame_id', keep='first')

            # Merge data on frame_id
            merged_df = pd.merge(mmwave_filtered, distance_filtered, on='frame_id', how='inner',
                                 suffixes=('', '_dist'))

            # Rename and create unified timestamp
            # Ensure 'timestamp' from mmwave is preserved and 'timestamp' from distance is 'timestamp_dist'
            if 'timestamp_dist' not in merged_df.columns and 'timestamp' in distance_filtered.columns:
                 # if suffixes didn't add _dist to distance_df's timestamp (e.g. if mmwave_df didn't have 'timestamp')
                 # This is a bit of a safeguard, merge suffixes should handle it.
                 pass # Suffixes should handle this. If not, more complex rename logic needed.

            merged_df.rename(columns={'timestamp': 'timestamp_mmwave'}, inplace=True) # timestamp from mmwave
            if 'timestamp_dist' not in merged_df.columns and 'timestamp' in distance_filtered.columns:
                # If 'timestamp_dist' was not created by merge, it means 'timestamp' from distance_df was not suffixed.
                # This might happen if 'timestamp' was not in mmwave_df originally.
                # However, we converted 'timestamp' in both, so 'timestamp' column should exist in both.
                # This path is less likely given the 'suffixes' argument.
                if 'timestamp_mmwave' in merged_df.columns and 'timestamp' in distance_filtered.columns:
                    # If both original timestamps are somehow named 'timestamp' after merge (unlikely with suffixes)
                    # This part might need adjustment based on actual merge output if suffixes don't work as expected
                    # For now, assuming suffixes=('', '_dist') works.
                    pass


            # Create the primary timestamp column from mmwave data
            merged_df['timestamp'] = merged_df['timestamp_mmwave']


            # Initialize loop_id and state machine
            merged_df['loop_id'] = 0
            current_loop_id = 0
            state = 'WAITING_FOR_BOX1'
            loop_ids = []

            # Check if necessary columns for loop detection exist
            if box1_col not in merged_df.columns or box5_col not in merged_df.columns:
                print(f"Missing column '{box1_col}' or '{box5_col}' in merged data for: {current_processing_folder}. Skipping loop ID generation.")
            else:
                for row in merged_df.itertuples(index=False):
                    # No try-except needed here if we checked columns existence above
                    is_box1_active = getattr(row, box1_col) == 1
                    is_box5_active = getattr(row, box5_col) == 1

                    if state == 'WAITING_FOR_BOX1':
                        if is_box1_active:
                            current_loop_id += 1
                            state = 'WAITING_FOR_BOX5'
                            loop_ids.append(current_loop_id)
                        else:
                            loop_ids.append(current_loop_id) # Append 0 if no loop started yet
                    elif state == 'WAITING_FOR_BOX5':
                        loop_ids.append(current_loop_id)
                        if is_box5_active:
                            state = 'WAITING_FOR_BOX1'
                    else: # Should not happen with defined states
                        loop_ids.append(current_loop_id)

                if len(loop_ids) == len(merged_df):
                    merged_df['loop_id'] = loop_ids
                else:
                    print(f"Warning: Length mismatch for loop_ids in {current_processing_folder}. Loop IDs not assigned correctly.")


            # Save the final merged and labeled DataFrame
            merged_df.to_csv(merged_path, index=False)
            print(f"[✓] Saved merged and labeled data to: {merged_path}")

        except pd.errors.EmptyDataError:
            print(f"[!] Empty CSV file encountered in {current_processing_folder}. Skipping.")
        except FileNotFoundError:
            # This should ideally not happen due to the check at the beginning of the loop,
            # but good for robustness if files are deleted during script execution.
            print(f"[!] File not found error during processing of {current_processing_folder}. This is unexpected.")
        except Exception as e:
            print(f"[!] Error processing folder {current_processing_folder}: {e}")
            import traceback
            traceback.print_exc() # For more detailed error information

print("\nProcessing complete.")